# 02 - Service B (Semantic Search with Chroma Persistence)

This notebook builds and validates the semantic retrieval service.

## Objectives
- Load and chunk source documents.
- Build embeddings and persist vectors in ChromaDB.
- Retrieve top-k context for user questions.
- Generate grounded answers with source evidence.

## Design Notes (Service B)

### Data source choice
- To keep this simple and reproducible, this notebook indexes a small local corpus from this repo:
  - `README.md`
  - `02_activities/assignment_2.md`
  - `05_src/assignment_chat/plan.md`
  - `05_src/assignment_chat/solution.md`

### Retrieval approach
- We use **semantic retrieval** with embeddings + Chroma PersistentClient.
- Chunks and metadata are stored once and reused across runs.
- Answers are generated from retrieved chunks and include source references.

### Why this meets assignment requirements
- Uses semantic search with Chroma persistence.
- Runs locally with repository data (well under size limits).
- Produces grounded responses instead of unsupported free-form answers.

# Load Secrets

In [5]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [ ]:
# Environment + model setup (aligned with assignment_1.ipynb)
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

API_GATEWAY_KEY = os.getenv("API_GATEWAY_KEY") or ""
client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": API_GATEWAY_KEY},
)
MODEL_NAME = "gpt-4o"
EMBEDDING_MODEL = "text-embedding-3-small"

# Runtime flags (simple behavior)
# 1) Default is local run only.
# 2) Set USE_OPENAI_BACKEND=True to run main flow with OpenAI.
# 3) Set RUN_COMPARISON=True to run both local and OpenAI side-by-side.
USE_OPENAI_BACKEND = False
RUN_COMPARISON = False

DEFAULT_BACKEND = "openai" if USE_OPENAI_BACKEND else "local"

# Notebook-specific imports
from typing import List, Dict, Any
import chromadb
from chromadb.api.models.Collection import Collection
from sentence_transformers import SentenceTransformer

# Chroma persistence base directory (inside assignment_chat folder)
CHROMA_BASE_DIR = Path.cwd() / "chroma_store" / "service_b"
COLLECTION_BASE_NAME = "assignment_semantic_chunks"

# Lazy-loaded local embedding fallback model
_local_embedder = None

In [7]:
# Chunking + indexing pipeline
from uuid import uuid4

def _find_repo_root(start: Path) -> Path:
    """Walk upward until we find repo markers."""
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "02_activities").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root from current working directory.")

def _simple_chunk(text: str, chunk_size: int = 900, overlap: int = 150) -> List[str]:
    """Character-based chunker with overlap to preserve context continuity."""
    clean = " ".join(text.split())
    if not clean:
        return []
    chunks = []
    start = 0
    while start < len(clean):
        end = min(len(clean), start + chunk_size)
        chunks.append(clean[start:end])
        if end == len(clean):
            break
        start = max(0, end - overlap)
    return chunks

def _embedding_backend_available(backend: str) -> bool:
    return backend == "local" or (backend == "openai" and bool(API_GATEWAY_KEY))

def _embed_texts(texts: List[str], backend: str, batch_size: int = 32) -> List[List[float]]:
    """Create embeddings with the selected backend."""
    global _local_embedder

    if backend == "openai":
        vectors: List[List[float]] = []
        for batch_start in range(0, len(texts), batch_size):
            batch = texts[batch_start:batch_start + batch_size]
            result = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
            vectors.extend([item.embedding for item in result.data])
        return vectors

    if _local_embedder is None:
        _local_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    encoded = _local_embedder.encode(texts, normalize_embeddings=True)
    return [vec.tolist() for vec in encoded]

def load_documents() -> List[Dict[str, Any]]:
    """Load a small local corpus for semantic retrieval."""
    repo_root = _find_repo_root(Path.cwd())
    candidates = [
        repo_root / "README.md",
        repo_root / "02_activities" / "assignment_2.md",
        repo_root / "05_src" / "assignment_chat" / "plan.md",
        repo_root / "05_src" / "assignment_chat" / "solution.md",
    ]

    docs: List[Dict[str, Any]] = []
    for path in candidates:
        if path.exists():
            text = path.read_text(encoding="utf-8", errors="ignore")
            source_path = str(path.relative_to(repo_root)).replace("\\", "/")
            docs.append({"source": source_path, "text": text})

    if not docs:
        raise FileNotFoundError("No source documents found for semantic indexing.")
    return docs

def _store_path_for_backend(backend: str) -> Path:
    return CHROMA_BASE_DIR / backend

def _collection_name_for_backend(backend: str) -> str:
    return f"{COLLECTION_BASE_NAME}_{backend}"

def build_or_load_index(backend: str) -> Collection:
    """Build or load a persistent Chroma collection for the selected backend."""
    if not _embedding_backend_available(backend):
        raise RuntimeError(f"Backend '{backend}' is not available in current environment.")

    store_path = _store_path_for_backend(backend)
    store_path.mkdir(parents=True, exist_ok=True)
    db = chromadb.PersistentClient(path=str(store_path))
    collection_name = _collection_name_for_backend(backend)
    collection = db.get_or_create_collection(name=collection_name)

    if collection.count() > 0:
        print(f"Loaded existing collection with {collection.count()} chunks (backend: {backend}).")
        return collection

    docs = load_documents()
    chunk_texts: List[str] = []
    metadatas: List[Dict[str, Any]] = []
    ids: List[str] = []

    for doc in docs:
        chunks = _simple_chunk(doc["text"], chunk_size=900, overlap=150)
        for idx, chunk in enumerate(chunks):
            chunk_texts.append(chunk)
            metadatas.append({"source": doc["source"], "chunk_index": idx})
            ids.append(str(uuid4()))

    vectors = _embed_texts(chunk_texts, backend=backend)
    collection.add(ids=ids, documents=chunk_texts, metadatas=metadatas, embeddings=vectors)
    print(f"Indexed {len(chunk_texts)} chunks into persistent Chroma store (backend: {backend}).")
    return collection

# Build active collections once and reuse
COLLECTIONS: Dict[str, Collection] = {}
COLLECTIONS[DEFAULT_BACKEND] = build_or_load_index(DEFAULT_BACKEND)

# Optional comparison setup: build the second backend collection too.
if RUN_COMPARISON:
    other_backend = "openai" if DEFAULT_BACKEND == "local" else "local"
    if _embedding_backend_available(other_backend):
        COLLECTIONS[other_backend] = build_or_load_index(other_backend)
    else:
        print(f"Comparison backend '{other_backend}' is unavailable; comparison will run only on '{DEFAULT_BACKEND}'.")

Loaded existing collection with 51 chunks (backend: local).


In [8]:
# Retrieval + grounded answer generation
def semantic_query(question: str, top_k: int = 4, backend: str = None) -> Dict[str, Any]:
    """Answer a question using semantic retrieval over persisted Chroma chunks."""
    q = (question or "").strip()
    if not q:
        raise ValueError("Question cannot be empty.")

    selected_backend = backend or DEFAULT_BACKEND
    if selected_backend not in COLLECTIONS:
        if _embedding_backend_available(selected_backend):
            COLLECTIONS[selected_backend] = build_or_load_index(selected_backend)
        else:
            raise RuntimeError(f"Backend '{selected_backend}' is unavailable.")

    collection = COLLECTIONS[selected_backend]
    q_vec = _embed_texts([q], backend=selected_backend)[0]
    result = collection.query(
        query_embeddings=[q_vec],
        n_results=max(1, top_k),
        include=["documents", "metadatas", "distances"],
    )

    docs = result.get("documents", [[]])[0]
    metas = result.get("metadatas", [[]])[0]
    dists = result.get("distances", [[]])[0]

    if not docs:
        return {
            "backend": selected_backend,
            "answer": "I could not find relevant evidence in the indexed documents. Please rephrase your question.",
            "sources": [],
            "retrieved_chunks": [],
        }

    evidence_lines = []
    source_set = []
    for doc, meta, dist in zip(docs, metas, dists):
        source = meta.get("source", "unknown")
        if source not in source_set:
            source_set.append(source)
        evidence_lines.append(f"- ({dist:.4f}) [{source}] {doc[:220]}...")

    answer = None
    if selected_backend == "openai" and API_GATEWAY_KEY:
        context_block = "\n\n".join([f"Source: {m.get('source', 'unknown')}\n{d}" for d, m in zip(docs, metas)])
        prompt = f"""
You are a retrieval assistant. Answer the user question using ONLY the provided context.
If the context is insufficient, say so clearly.

Question: {q}

Context:
{context_block}
""".strip()

        try:
            llm_resp = client.responses.create(
                model=MODEL_NAME,
                input=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            answer = llm_resp.output_text.strip()
        except Exception:
            answer = None

    if not answer:
        # Deterministic fallback grounded directly in top retrieved chunks.
        summary_points = [doc[:200].strip() for doc in docs[:2]]
        answer = "Based on the top retrieved evidence: " + " ".join(summary_points)

    return {
        "backend": selected_backend,
        "answer": answer,
        "sources": source_set,
        "retrieved_chunks": evidence_lines,
    }

def run_comparison(question: str, top_k: int = 4) -> Dict[str, Dict[str, Any]]:
    """Run the same query on all available backends for quick comparison."""
    outputs: Dict[str, Dict[str, Any]] = {}
    for backend_name in ["local", "openai"]:
        if _embedding_backend_available(backend_name):
            outputs[backend_name] = semantic_query(question, top_k=top_k, backend=backend_name)
    return outputs

# Demo queries to validate behavior
demo_questions = [
    "What are the key requirements for Assignment 2?",
    "How does the plan handle guardrails and restricted topics?",
    "What does the architecture say about component responsibilities?",
]

for question in demo_questions:
    print(f"\n=== Question: {question}")

    if RUN_COMPARISON:
        compare_outputs = run_comparison(question, top_k=4)
        for backend_name, output in compare_outputs.items():
            print(f"\n[{backend_name.upper()}] Answer: {output['answer']}")
            print(f"[{backend_name.upper()}] Sources: {output['sources']}")
            print(f"[{backend_name.upper()}] Top evidence:")
            for line in output["retrieved_chunks"][:2]:
                print(line)
    else:
        output = semantic_query(question, top_k=4, backend=DEFAULT_BACKEND)
        print(f"Backend: {output['backend']}")
        print("Answer:", output["answer"])
        print("Sources:", output["sources"])
        print("Top evidence:")
        for line in output["retrieved_chunks"][:2]:
            print(line)


=== Question: What are the key requirements for Assignment 2?
Backend: local
Answer: Based on the top retrieved evidence: gging skills. They will cover foundational skills and will extend to advanced concepts. We recommend that you attempt all assignments and submit your work, even if it is incomplete (partial submission # Assignment 2 Solution Architecture (`assignment_chat`) ## 1. Solution Objective Design and implement a chat-based AI system that satisfies all assignment constraints using a practical, low-risk arch
Sources: ['README.md', '05_src/assignment_chat/solution.md']
Top evidence:
- (0.8806) [README.md] gging skills. They will cover foundational skills and will extend to advanced concepts. We recommend that you attempt all assignments and submit your work, even if it is incomplete (partial submissions will earn partial ...
- (1.0088) [05_src/assignment_chat/solution.md] # Assignment 2 Solution Architecture (`assignment_chat`) ## 1. Solution Objective Design and implement a 

## Validation Checklist
- [x] Vector store persists across notebook runs.
- [x] Retrieval returns relevant chunks for test questions.
- [x] Final answer is grounded and cites retrieved sources.

## Notes
- Default execution is local (`USE_OPENAI_BACKEND=False`, `RUN_COMPARISON=False`).
- Set `USE_OPENAI_BACKEND=True` to run the main flow with OpenAI embeddings/synthesis.
- Set `RUN_COMPARISON=True` to run both local and OpenAI backends side-by-side for the same question.
- Chroma stores are separated by backend to keep dimensions consistent and avoid mismatch errors.
- This notebook keeps implementation intentionally simple and aligned with Assignment 2 requirements.